# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an interactive, reproducible workflow for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. You'll explore metadata, record sets, and fields, extract records by `@id`, perform basic EDA, and visualize data.

### Dataset Source
The FAIR^2 Croissant dataset schema is available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if it's not available
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. We'll define the dataset Croissant schema URL and extract metadata details like name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata
print(f"Dataset name: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")
print(f"Dataset version: {dataset.metadata.version if hasattr(dataset.metadata, 'version') else 'N/A'}")
print(f"Dataset license: {dataset.metadata.license if hasattr(dataset.metadata, 'license') else 'N/A'}")

## 2. Data Overview

Review available record sets, fields, and their `@id` values using the `dataset.metadata`. All `@id` fields help uniquely identify elements for extraction and analysis.

Let's enumerate all record sets, their fields, and column mapping, referencing them only by their `@id` as required.

In [ ]:
# Helper to pretty-print available record sets, fields, and columns by @id
def summarize_metadata(ds):
    print("Available Record Sets (by @id):\n")
    for recset in ds.metadata.record_sets:
        print(f" - {recset.id} (name: {recset.name})")
        print("   Fields (by @id):")
        for field in recset.fields:
            print(f"     - {field.id} (name: {field.name}, dataType: {getattr(field, 'data_type', None)})")
        print("   Columns (by @id and field mapping):")
        for col in getattr(recset, 'columns', []):
            print(f"     - {col.id} (name: {col.name}) -> field: {col.field}")
        print()

# RecordSet overview (by @id)
summarize_metadata(dataset)

## 3. Data Extraction

Let's load data from one or more record sets into pandas DataFrames using their correct `@id` as referenced above.

Reference all record sets using their `@id`, extract the records and display the columns and a preview. Modify `<record_set_id>` to experiment with other sets.

In [ ]:
# List all available record set @ids
record_sets = [recset.id for recset in dataset.metadata.record_sets]

# Load each record set as a DataFrame
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id} with {len(df)} records.")

# Example: Show columns and the first 5 rows for the primary protein data record set
# (Replace with actual @id found from summary above)
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"Columns for record set '{main_record_set_id}':\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)

Let's analyze the data from one of the record sets. We'll select a numeric field (by its `@id`) for filtering and normalization. You may need to adjust the field IDs to those present in your dataset and check the relevant columns for real numeric fields.

In [ ]:
import numpy as np

# Pick one record set (update the variable to the relevant @id as needed; we use the first one)
record_set_id = main_record_set_id  # as above
df = dataframes[record_set_id]

# Guess a likely numeric field (replace this @id/column as needed)
numeric_candidate_columns = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric candidate columns: {numeric_candidate_columns}")

# Select the first numeric field available for demonstration
if numeric_candidate_columns:
    numeric_field = numeric_candidate_columns[0]
else:
    numeric_field = df.columns[0]  # fallback

print(f"Using numeric field: {numeric_field}")

# Example filter: Filter records where numeric_field > threshold
threshold = df[numeric_field].quantile(0.7) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field]) else df
print(f"Filtered records with {numeric_field} > {threshold} (showing first 5 records):")
display(filtered_df.head())

# Normalize the numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print("Normalized values (first 5):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping if any categorical column is present
group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
group_field = group_fields[0] if group_fields else None
if group_field and pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean of {numeric_field} by {group_field} (first 5):")
    display(grouped_df.head())

## 5. Visualization

Let's visualize distributions of the selected numeric field and its normalized values, and show a boxplot by a categorical variable if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(10, 5))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# Normalized field (if available)
if f"{numeric_field}_normalized" in filtered_df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, color='green')
    plt.title(f"Distribution of Normalized {numeric_field}")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by category if available
if group_field:
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, you explored a mass spectrometry human protein dataset by loading its Croissant schema, reviewing available record sets and fields (referenced by `@id`), and loading records via `mlcroissant`. Using pandas and seaborn, you conducted basic EDA and visualizations. Modify fields and record sets by their `@id` to adapt this notebook to other Croissant datasets or perform deeper analysis!